<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c36-header.png" alt="Nextia Learning · Text Classification and Routing: Send Each Message to the Right Place" width="100%">

# Solution: Case study: sort requests with few labels

**[Case study: sort requests with few labels](https://learning.nextia-ai.com/courses/text-routing/m10/sort-requests-with-few-labels/)** · Text Classification and Routing: Send Each Message to the Right Place · Module 10, case study 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** start an English and French assistant with no labels, and find out how many labels it needs: 10 per intent, then 50, then all. You will compare TF-IDF, embeddings, SetFit, a zero-shot model and a recorded LLM on one measure, draw the learning curve in both languages, test whether English labels are enough for French, and decide when to stop labelling.

| | |
|---|---|
| **Time** | About 2.5 hours with the lesson page. The notebook runs in about 3 minutes on a laptop after the install and the downloads. |
| **Needs** | An internet connection for the setup cells: MASSIVE (a 40 MB archive, of which we keep 16 MB), the course's files (2 MB) and two models (about 470 MB and 320 MB, once). No account and no key: the LLM's answers were recorded. |
| **You should know** | Few labels with embeddings and SetFit, from [Few labels with embeddings](https://learning.nextia-ai.com/courses/text-routing/m04/few-labels-with-embeddings/); zero-shot models, from [Zero-shot without an LLM](https://learning.nextia-ai.com/courses/text-routing/m04/zero-shot-without-an-llm/); the review threshold, from [Decide, review or send to a person](https://learning.nextia-ai.com/courses/text-routing/m08/decide-review-or-send-to-a-person/). |
| **Data** | MASSIVE 1.1 by Amazon, CC BY 4.0, built from SLURP (CC BY 4.0). Details in `dataset.md` next to this notebook. |
| **Tested** | Python 3.12 with pandas 3.1.1, scikit-learn 1.9.1 and matplotlib 3.11.2, sentence-transformers 6.2.0, SetFit 1.2.0, GLiClass 0.1.22, Transformers 5.22.2 and PyTorch 2.15.0 (CPU), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/text-routing/m10/sort-requests-with-few-labels/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C36/M10-L03-sort-requests-with-few-labels/sort-requests-with-few-labels-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, scikit-learn 1.9.1, sentence-transformers 6.2.0, torch 2.15.0, setfit 1.2.0, gliclass 0.1.22, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time. PyTorch is large (about 200 MB to download): the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scikit-learn==1.9.1 matplotlib==3.11.2 sentence-transformers==6.2.0 torch==2.15.0 setfit==1.2.0 gliclass==0.1.22
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import sentence_transformers; print("sentence-transformers", sentence_transformers.__version__)
import torch; print("torch", torch.__version__)
import setfit; print("setfit", setfit.__version__)
import gliclass; print("gliclass", gliclass.__version__)

## Setup: get the data and the course's files

Run the next cell. It downloads into a folder `massive-requests`:

- Amazon's MASSIVE archive (40 MB). It checks the archive's checksum, keeps only the English and French files (16 MB) and deletes the rest;
- the course's files: the 13 intent descriptions, the recorded LLM answers and the helper `routing.py`.

It checks each file's checksum. You should see `Ready: 6 files in massive-requests/`. If you run the cell again, it uses the files that are already there.

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `massive-requests`, then run the cell again.

**Source and licence.** "MASSIVE 1.1" by FitzGerald et al. (Amazon), [github.com/alexa/massive](https://github.com/alexa/massive), licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). It translates SLURP by Bastianelli et al. (CC BY 4.0). Crowd workers wrote the requests for a voice assistant: no real customers. `dataset.md` lists what we changed.

In [ ]:
# Setup: download MASSIVE (English and French) and the case study's files into massive-requests/, and check them.
import hashlib, os, shutil, subprocess, tarfile, urllib.request
from pathlib import Path

ARCHIVE = ["https://amazon-massive-nlu-dataset.s3.amazonaws.com/amazon-massive-dataset-1.1.tar.gz", "4cba5faa11c71437928e17cb1b9b3d8b8e727e7ea363a3a9a8045e19c0491577"]
UPSTREAM = {
    "data/en-US.jsonl": "c70f75c6a543a26e249ec383df67733ad9b1066f6c0406c2e04a3f03356e407e",
    "data/fr-FR.jsonl": "f9bf3db170ad415b389e4c9594dd0f8f80c38188143e05cc4459a6fa7df7cf49",
}
FILES = {
    "data/intents.csv": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C36/M10-L03-sort-requests-with-few-labels/data/intents.csv",
        "498eb9809babdc757b75f5fb0c68f7acc6751263214715ebe984f2e4514dd16d"
    ],
    "recordings/llm-few-shot.jsonl": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C36/M10-L03-sort-requests-with-few-labels/recordings/llm-few-shot.jsonl",
        "ac33f7ca20c4e57bcd2d2dd0f9549676561b6d456997af0c1963924d13c71777"
    ],
    "recordings/llm-zero-shot.jsonl": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C36/M10-L03-sort-requests-with-few-labels/recordings/llm-zero-shot.jsonl",
        "32055a8ab702f070f48358a9340be9db32f8b4564451fe00a9ce21f94e946476"
    ],
    "routing.py": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C36/routing.py",
        "8de6562d53415e5c3d8e74f44df9e8838cb71085011b29244187bde5a6f4d93c"
    ],
}
HERE = Path.cwd()
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("massive-requests").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

# 1. MASSIVE: Amazon's archive (40 MB, 52 languages); we keep only en-US and fr-FR.
if not all((PROJECT / n).exists() and sha256(PROJECT / n) == h for n, h in UPSTREAM.items()):
    tar_path = PROJECT / "massive-1.1.tar.gz"
    download(ARCHIVE[0], tar_path)
    if sha256(tar_path) != ARCHIVE[1]:
        raise SystemExit("The MASSIVE archive has the wrong checksum. Delete the folder massive-requests and run again.")
    with tarfile.open(tar_path) as tar:
        for name in UPSTREAM:
            member = tar.getmember("1.1/" + name)
            with tar.extractfile(member) as src, open(PROJECT / name, "wb") as dst:
                shutil.copyfileobj(src, dst)
    tar_path.unlink()
    for name, expected in UPSTREAM.items():
        if sha256(PROJECT / name) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder massive-requests and run again.")

# 2. The course's files: the intent descriptions, the recorded LLM answers and routing.py.
for name, (url, expected) in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    local = HERE / name  # a copy next to the notebook, if there is one
    if local.exists():
        shutil.copy(local, path)
    else:
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder massive-requests and run again.")
os.chdir(PROJECT)
print(f"Ready: {len(UPSTREAM) + len(FILES)} files in {PROJECT.name}/")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, PyTorch and the course helper as `rt`, and defines the check helpers. They tell you if your answer is right, without showing it:

- `expect(what, yours, expected)` compares one of your numbers with the expected one.
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the libraries, the course helper, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import routing as rt
import time, torch
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The data

One row is one request and its intent. We keep 4 of MASSIVE's 18 scenarios: alarms, calendar, email and lists. They are the requests of one product, a calendar, email and list assistant, with 13 intents.

> **Check.** You should see 10,188 requests in 13 intents: in each language, 3,570 train, 613 validation and 911 test.

In [ ]:
import json, os
os.environ["TQDM_DISABLE"] = "1"   # no progress bars from the model packages
# The case study uses 4 of MASSIVE's 18 scenarios: the requests of a calendar, email and list assistant.
SCENARIOS = ["alarm", "calendar", "email", "lists"]

def read(locale, language):
    with open(f"data/{locale}.jsonl", encoding="utf-8") as f:
        df = pd.DataFrame([json.loads(line) for line in f])
    df = df[df.scenario.isin(SCENARIOS)]
    return pd.DataFrame({"message_id": f"MA-{language}-" + df.id, "id": df.id, "language": language,
                         "split": df.partition.replace({"dev": "validation"}), "scenario": df.scenario,
                         "intent": df.intent, "text": df.utt})

requests = pd.concat([read("en-US", "en"), read("fr-FR", "fr")], ignore_index=True)
counts = requests.groupby(["split", "language"]).size().unstack()
print(f"{len(requests):,} requests in {requests.intent.nunique()} intents, {len(SCENARIOS)} scenarios")
counts.loc[["train", "validation", "test"]]

## 2. First checks

Each French request is a translation of the English request with the same `id`. The next cell checks that, looks for repeated texts, and counts the requests per intent.

> **Check.** 100% of the pairs have the same intent. The smallest intent is `email_addcontact`, with 54 training requests in each language.

In [ ]:
en = requests[requests.language == "en"]
fr = requests[requests.language == "fr"].set_index("id")
same = (en.set_index("id").intent == fr.intent).mean()
print(f"English and French rows with the same id and the same intent: {same:.0%} (each French request is a translation)")
for lang, part in requests.groupby("language"):
    train_text = set(part.text[part.split == "train"])
    test_part = part[part.split == "test"]
    print(f"{lang}: missing texts {part.text.isna().sum()} | repeated texts in train "
          f"{part.text[part.split == 'train'].duplicated().sum()} | test texts also in train "
          f"{test_part.text.isin(train_text).sum()} | median words {part.text.str.split().str.len().median():.0f}")
per_intent = en[en.split == "train"].intent.value_counts()
print(f"train requests per intent (each language): from {per_intent.min()} ({per_intent.idxmin()}) "
      f"to {per_intent.max()} ({per_intent.idxmax()})")
print(requests[requests.id == "1"].sort_values(["id", "language"])[["id", "language", "intent", "text"]]
      .to_string(index=False))

## 3. The case study's helpers and the cost table

`score()` returns one row of the comparison table, with the course's columns, from `routing.py`. The 13 intents take the place of the routes. The costs are **assumptions** for this app (the lesson page says why):

- the app asks the user to choose (a "review"): 0.10 dollars;
- a wrong action: 0.50 dollars (the user sees it and types again);
- a wrong action on `alarm_set` or `calendar_set`: 3 dollars (some users do not notice, and miss the alarm or the meeting).

> **Check.** "Everything to the user" (always ask) costs 100 dollars per 1,000 requests.

In [ ]:
# The case study's helpers: the course's scoring (routing.py) applied to 13 intents and this app's costs.
intents = pd.read_csv("data/intents.csv")
INTENTS = list(intents.intent)
PRIORITY = dict(zip(intents.intent, intents.priority))
rt.ROUTES = INTENTS                         # routing.py scores these 13 "routes"
# The cost of each outcome in US dollars: assumptions for this app (the page says why).
COSTS = {"review": 0.10,                    # the app asks the user to choose: a few seconds of their time
         "wrong_queue": 0.50,               # a wrong action: the user sees it and types again
         "wrong_queue_urgent": 3.00,        # a missed "set" (alarm or event): some users do not notice
         "missed_account_takeover": 0.0, "needs_person_sent_to_team": 0.0, "team_message_sent_to_person": 0.0}
CPU_HOUR = 0.05                             # dollars per CPU-hour (the course's assumption)
RARE = "email_addcontact"                   # the smallest intent: 54 training requests per language
train, validation, test = (requests[requests.split == s].reset_index(drop=True) for s in ("train", "validation", "test"))

def truth_of(part):
    """The columns that routing.py needs: route (the intent), topic, priority and language."""
    return pd.DataFrame({"route": part.intent.values, "topic": part.intent.values, "language": part.language.values,
                         "priority": part.intent.map(PRIORITY).values}, index=part.index)

def score(name, part, predicted, labels=None, run_per_1000=0.0, ms=None):
    """One row of the comparison table. A prediction can be "review": the app asks the user to choose."""
    row = rt.score(name, truth_of(part), list(predicted), run_per_1000=run_per_1000, ms_per_message=ms, costs=COSTS)
    predicted = np.asarray(predicted)
    rare = (part.intent == RARE).to_numpy()
    return {"method": name, "labels_per_intent": labels, "accuracy": row["accuracy"], "macro_f1": row["macro_f1"],
            "accuracy_en": row["accuracy_en"], "accuracy_fr": row["accuracy_fr"],
            "rare_recall": round(float((predicted[rare] == RARE).mean()), 3), "review_share": row["review_share"],
            "mistakes_per_1000": row["mistakes_per_1000"], "run_per_1000": row["run_per_1000"],
            "total_per_1000": row["total_per_1000"], "ms_per_message": row["ms_per_message"]}

def table(rows):
    """The rows as a table, cheapest total first."""
    return pd.DataFrame(rows).sort_values("total_per_1000").reset_index(drop=True)

def cpu_cost(ms):
    return ms / 1000 / 3600 * CPU_HOUR * 1000   # dollars per 1,000 requests at `ms` milliseconds each

def ms_per_message(predict, texts, n=100):
    t0 = time.perf_counter()
    for t in texts[:n]:
        predict([t])
    return (time.perf_counter() - t0) / n * 1000

table([score("everything to the user", validation, ["review"] * len(validation), 0)])

## 4. Draw the labels

A labeller labels 10 requests per intent: 130 in each language. `draw(k)` draws the same request ids in both languages, so the French labels are the translations of the English ones. The seed is fixed, so you get the same requests as the course.

In [ ]:
def draw(k, repeat=0):
    """The ids of k training requests per intent (all of them if an intent has fewer), at random.
    The same ids in both languages: the French labeller labels the translations of the same requests."""
    pool = train[train.language == "en"]
    rng = np.random.default_rng([36, k, repeat])
    ids = []
    for intent in INTENTS:
        mine = pool.id[pool.intent == intent].to_numpy()
        ids.extend(rng.choice(mine, size=min(k, len(mine)), replace=False))
    return set(ids)

ALL = set(train.id)
def labelled(ids, language):
    return train[train.id.isin(ids) & (train.language == language)]

ids10 = draw(10)
print(len(ids10), "requests per language with 10 labels per intent:")
print(labelled(ids10, "en").head(3)[["intent", "text"]].to_string(index=False))

## 5. The baseline: TF-IDF with 10 labels per intent

The course project's TF-IDF pipeline, one model per language, trained on the 130 labelled requests of that language. We compare on the 1,226 validation requests (both languages). The test requests stay closed until section 14.

> **Check.** You should see an accuracy of about 0.69. Everything to the user costs less.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion, make_pipeline

def tfidf_lr():
    """The course project's TF-IDF + logistic regression (larkroute.tfidf): words and word pairs, plus letter groups."""
    words = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, strip_accents="unicode")
    letters = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True)
    return make_pipeline(FeatureUnion([("words", words), ("letters", letters)]), LogisticRegression(C=10, max_iter=3000))

def per_language(fit_predict, ids, part):
    """Train one model per language on that language's labels; predict the part (validation or test)."""
    out = pd.Series(index=part.index, dtype=object)
    conf = pd.Series(index=part.index, dtype=float)
    for lang in ("en", "fr"):
        rows = part.language == lang
        pred, c = fit_predict(labelled(ids, lang), part[rows])
        out[rows], conf[rows] = pred, c
    return out, conf

def tfidf_fit_predict(lab, target):
    m = tfidf_lr().fit(lab.text, lab.intent)
    p = m.predict_proba(target.text)
    return m.classes_[p.argmax(1)], p.max(1)

tfidf10, _ = per_language(tfidf_fit_predict, ids10, validation)
rows = [score("everything to the user", validation, ["review"] * len(validation), 0),
        score("TF-IDF + LR", validation, tfidf10, 10)]
table(rows)

## 6. Embeddings

The course's older embedding model turns each request into 384 numbers. It is multilingual: one model for English and French. The cell embeds all 10,188 requests: about 20 seconds on a laptop, plus the model download the first time.

In [ ]:
from sentence_transformers import SentenceTransformer
torch.set_num_threads(4)
# The course's older embedding model, frozen. It is multilingual: one model for English and French.
MODEL, REV = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", "e8f8c211226b894fcb81acc59f3b34ba3efd5f42"
embedder = SentenceTransformer(MODEL, revision=REV, device="cpu")
t0 = time.perf_counter()
E = {}
for part in (train, validation, test):
    for mid, vec in zip(part.message_id, embedder.encode(list(part.text), batch_size=64, normalize_embeddings=True)):
        E[mid] = vec
print(f"{len(E):,} requests embedded in {time.perf_counter() - t0:.0f} s; {len(vec)} numbers each")

def vectors(part):
    return np.stack([E[m] for m in part.message_id])

## 7. Three methods with 10, 50 and all labels per intent

TF-IDF, the nearest centre and logistic regression on the embeddings, each trained on 10, 50 and all labels per intent.

> **Predict.** With 10 labels per intent, which method wins? And with all labels?

> **Check.** With 10 labels, embeddings + LR reaches about 0.749. With all labels, TF-IDF reaches about 0.911.

In [ ]:
def emb_lr_fit_predict(lab, target):
    m = LogisticRegression(C=10, max_iter=3000).fit(vectors(lab), lab.intent)
    p = m.predict_proba(vectors(target))
    return m.classes_[p.argmax(1)], p.max(1)

def centre_fit_predict(lab, target):
    """Nearest centre: the average embedding of each intent; the most similar centre wins."""
    names = sorted(lab.intent.unique())
    centres = np.stack([vectors(lab[lab.intent == i]).mean(axis=0) for i in names])
    centres /= np.linalg.norm(centres, axis=1, keepdims=True)
    sims = vectors(target) @ centres.T
    return np.array(names)[sims.argmax(1)], sims.max(1)

METHODS = {"TF-IDF + LR": tfidf_fit_predict, "embeddings + nearest centre": centre_fit_predict,
           "embeddings + LR": emb_lr_fit_predict}
pred_val, conf_val = {}, {}
for k, ids in [(10, ids10), (50, draw(50)), ("all", ALL)]:
    for name, fp in METHODS.items():
        pred_val[name, k], conf_val[name, k] = per_language(fp, ids, validation)
        if (name, k) != ("TF-IDF + LR", 10):
            rows.append(score(name, validation, pred_val[name, k], k))
table(rows)[["method", "labels_per_intent", "accuracy", "accuracy_en", "accuracy_fr", "rare_recall", "total_per_1000"]]

## 8. SetFit with 10 labels per intent

SetFit fine-tunes the embedding model itself on pairs of the 130 labelled requests, then trains logistic regression on top. It takes about a minute for both languages.

> **Check.** SetFit reaches about 0.84 with only 10 labels per intent, close to logistic regression with 50 labels (0.852). SetFit's training is not exactly the same on every computer, so the third decimal can differ.

In [ ]:
import tempfile
from datasets import Dataset
from setfit import SetFitModel, Trainer, TrainingArguments
# SetFit: fine-tune the embedding model itself on pairs of the 130 labelled requests, then train LR on top.
# The same settings as the course's lesson "Few labels with embeddings". About 30 seconds per language.
setfit_models = {}
t0 = time.perf_counter()
for lang in ("en", "fr"):
    torch.manual_seed(36)
    lab = labelled(ids10, lang)
    sf = SetFitModel.from_pretrained(MODEL, revision=REV, device="cpu")
    args = TrainingArguments(output_dir=tempfile.mkdtemp(), batch_size=16, num_epochs=1, num_iterations=10, seed=36, save_strategy="no", logging_strategy="no", report_to="none")  # no checkpoints/ folder here
    Trainer(model=sf, args=args, train_dataset=Dataset.from_dict(
        {"text": list(lab.text), "label": list(lab.intent)})).train()
    setfit_models[lang] = sf
print(f"SetFit trained in {time.perf_counter() - t0:.0f} s for both languages")

def setfit_predict(part):
    pred, conf = pd.Series(index=part.index, dtype=object), pd.Series(index=part.index, dtype=float)
    for lang, sf in setfit_models.items():
        rows_l = part.language == lang
        p = np.asarray(sf.predict_proba(list(part.text[rows_l])))
        pred[rows_l], conf[rows_l] = sf.model_head.classes_[p.argmax(1)], p.max(1)
    return pred, conf

pred_val["SetFit", 10], conf_val["SetFit", 10] = setfit_predict(validation)
rows.append(score("SetFit", validation, pred_val["SetFit", 10], 10))
table(rows)[["method", "labels_per_intent", "accuracy", "accuracy_en", "accuracy_fr", "rare_recall", "total_per_1000"]]

## 9. The learning curve

How does the accuracy grow with the labels? The next cell trains the three methods on 5, 10, 20, 50 and 100 labels per intent, with 5 random draws each, and on all labels. It takes about 15 seconds.

> **Check.** In English, embeddings + LR with 10 labels per intent has a mean of about 0.7589; in French about 0.7135.

In [ ]:
# The learning curve: 5 random draws for each number of labels per intent (all labels: one "draw").
SIZES = [5, 10, 20, 50, 100, "all"]
curve = []
for k in SIZES:
    for repeat in range(1 if k == "all" else 5):
        ids = ALL if k == "all" else draw(k, repeat)
        for name, fp in METHODS.items():
            p, _ = per_language(fp, ids, validation)
            for lang in ("en", "fr"):
                rows_l = validation.language == lang
                curve.append({"k": k, "repeat": repeat, "method": name, "language": lang, "labels": len(ids),
                              "accuracy": float((p[rows_l] == validation.intent[rows_l]).mean())})
curve = pd.DataFrame(curve)
summary = curve.groupby(["method", "language", "k"], sort=False).accuracy.agg(["mean", "min", "max"]).round(3)
summary.unstack("k")["mean"]

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, lang in zip(axes, ("en", "fr")):
    for name in METHODS:
        ax.plot(range(len(SIZES)), summary.loc[(name, lang)]["mean"].values, marker="o", label=name)
    ax.set_xticks(range(len(SIZES)), [str(k) for k in SIZES])
    ax.set_title({"en": "English", "fr": "French"}[lang])
    ax.set_xlabel("labels per intent")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("validation accuracy (mean of 5 draws)")
axes[0].legend()
plt.show()

## 10. Your turn: are English labels enough for French?

Labels in French cost a French-speaking labeller. Train on the **English** labels only, and route the **French** validation requests. Compare with the same method trained on the French labels.

> **Predict.** Which method survives the change of language: TF-IDF, or the multilingual embeddings?

In [ ]:
# Do we need French labels? Train on English labels only, then route the French validation requests.
fr_val = validation[validation.language == "fr"]
transfer = []
for k, ids in [(10, ids10), (50, draw(50)), ("all", ALL)]:
    for name, fp in [("TF-IDF + LR", tfidf_fit_predict), ("embeddings + LR", emb_lr_fit_predict)]:
        english, _ = fp(labelled(ids, "en"), fr_val)
        french, _ = fp(labelled(ids, "fr"), fr_val)
        transfer.append({"labels_per_intent": k, "method": name,
                         "french_labels": round(float((french == fr_val.intent).mean()), 3),
                         "english_labels_only": round(float((english == fr_val.intent).mean()), 3)})
emb10_english_only = transfer[1]["english_labels_only"]
pd.DataFrame(transfer)

In [ ]:
expect('emb10_english_only', emb10_english_only, 0.737, tolerance=0.001)

The solution notebook prints the whole table: TF-IDF and embeddings, with 10, 50 and all labels per intent. TF-IDF trained on English falls to about 0.418 on French: it knows only the English words it saw. The embeddings trained on English stay close to the embeddings trained on French.

## 11. Zero-shot: no labels at all

GLiClass reads the request and the 13 descriptions of `data/intents.csv` together and scores each one. No labels. It takes about 30 seconds, plus the model download the first time.

> **Check.** GLiClass reaches about 0.396: far below every trained method.

In [ ]:
from gliclass import GLiClassModel, ZeroShotClassificationPipeline
from transformers import AutoTokenizer
# Zero-shot: no labels at all. GLiClass reads the request and the 13 descriptions together and scores each one.
name, rev = "knowledgator/gliclass-multilang-edge", "d16c08ef70547514081952104e6fc3d190d9ee39"
gmodel = GLiClassModel.from_pretrained(name, revision=rev).float()   # some weights are stored in bfloat16
gli = ZeroShotClassificationPipeline(gmodel, AutoTokenizer.from_pretrained(name, revision=rev),
                                     classification_type="multi-label", device="cpu")
DESC = dict(zip(intents.description, intents.intent))

def gliclass_scores(texts, batch_size=32):
    out = []
    for i in range(0, len(texts), batch_size):
        for res in gli(list(texts[i:i + batch_size]), list(DESC), threshold=0.0, batch_size=batch_size):
            out.append({DESC[r["label"]]: r["score"] for r in res})
    return pd.DataFrame(out)[INTENTS]

t0 = time.perf_counter()
G = gliclass_scores(list(validation.text))
gli_ms = ms_per_message(lambda t: gliclass_scores(t), list(validation.text), n=50)
print(f"GLiClass: {len(G):,} requests in {time.perf_counter() - t0:.0f} s; {gli_ms:.0f} ms per request alone")
pred_val["GLiClass zero-shot", 0], conf_val["GLiClass zero-shot", 0] = G.idxmax(axis=1), G.max(axis=1)
rows.append(score("GLiClass zero-shot", validation, pred_val["GLiClass zero-shot", 0], 0, cpu_cost(gli_ms), gli_ms))
table(rows)[["method", "labels_per_intent", "accuracy", "accuracy_en", "accuracy_fr", "rare_recall", "total_per_1000"]]

## 12. The hosted LLM, recorded

A hosted LLM (Azure chat-small) chose an intent for every validation and test request, once with the descriptions only (zero-shot), and once with the descriptions and the 8 most similar of the 130 labelled requests (few-shot). The next cell builds the few-shot prompt for one request, so you can read it. The answers were recorded: you replay them, with no key.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
# The prompts of the recorded LLM answers. The few-shot prompt adds the 8 most similar of the 130 labelled
# requests (10 per intent) in the same language.
listing = "\n".join(f"- {i}: {d}" for i, d in zip(intents.intent, intents.description))
SYSTEM = {
    "zero-shot": ("You sort the requests that people type into a calendar, email and list assistant, in English or "
                  f"French. Choose exactly one intent from this list:\n{listing}\n"
                  "Answer in JSON with the intent and your confidence from 0 to 1."),
    "few-shot": ("You sort the requests that people type into a calendar, email and list assistant, in English or "
                 f"French. Choose exactly one intent from this list:\n{listing}\n"
                 "You also get labelled examples of similar requests. "
                 "Answer in JSON with the intent and your confidence from 0 to 1."),
}
pickers = {}
for lang in ("en", "fr"):
    lab = labelled(ids10, lang).reset_index(drop=True)
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True).fit(lab.text)
    pickers[lang] = (lab, vec, vec.transform(lab.text))

def examples_for(text, lang, k=8):
    """The k labelled requests (of the 130) with the closest TF-IDF vectors (letters), with their intents."""
    lab, vec, X = pickers[lang]
    sims = (X @ vec.transform([text]).T).toarray().ravel()
    return [(lab.text[i], lab.intent[i]) for i in sorted(sims.argsort()[::-1][:k])]

def user_prompt(text, lang, mode):
    if mode == "zero-shot":
        return f"Request:\n{text}"
    shown = "\n".join(f"- {t!r} -> {i}" for t, i in examples_for(text, lang))
    return f"Labelled examples:\n{shown}\n\nRequest:\n{text}"

print(user_prompt(validation.text[0], "en", "few-shot"))

> **Check.** Zero-shot reaches about 0.927 with the descriptions only, more than TF-IDF with all labels. Few-shot reaches about 0.932. Each answer takes more than a second and costs a small fraction of a cent. MASSIVE is public, so the LLM may have seen these requests in its training: the lesson page discusses this.

In [ ]:
# Replay the recorded answers of the hosted LLM (chat-small): you need no key.
def recorded(mode, part):
    rec = pd.read_json(f"recordings/llm-{mode}.jsonl", lines=True).drop_duplicates("message_id")
    rec = rec.set_index("message_id").loc[part.message_id]
    pred = np.where(rec.valid, rec.intent, "review")
    return rec, pd.Series(pred, index=part.index), pd.Series(rec.confidence.values, index=part.index)

PRICE_IN, PRICE_OUT = 0.10, 0.50            # chat-small, dollars per 1M tokens (the course's price table)
for mode, k in [("zero-shot", 0), ("few-shot", 10)]:
    rec, pred, conf = recorded(mode, validation)
    run = (rec.tokens_in.mean() * PRICE_IN + rec.tokens_out.mean() * PRICE_OUT) / 1e6 * 1000
    llm_ms = rec.latency_s.median() * 1000
    pred_val[f"LLM {mode}", k], conf_val[f"LLM {mode}", k] = pred, conf
    rows.append(score(f"LLM {mode} (recorded)", validation, pred, k, run, llm_ms))
    print(f"{mode}: {rec.tokens_in.mean():.0f} tokens in, {rec.tokens_out.mean():.0f} out, "
          f"{run:.3f} dollars per 1,000, median {llm_ms:,.0f} ms, valid answers {rec.valid.mean():.1%}")

## 13. The comparison table and the thresholds

The next cell adds the speed (one request alone) and the computer cost, and shows every row, cheapest first.

In [ ]:
# Speed: milliseconds for one request alone, for the models trained on all labels (English).
lab_all = labelled(ALL, "en")
tf_model = tfidf_lr().fit(lab_all.text, lab_all.intent)
lr_model = LogisticRegression(C=10, max_iter=3000).fit(vectors(lab_all), lab_all.intent)
texts = list(validation.text[validation.language == "en"])
ms = {"TF-IDF + LR": ms_per_message(tf_model.predict_proba, texts),
      "embeddings + LR": ms_per_message(lambda t: lr_model.predict_proba(embedder.encode(t, normalize_embeddings=True)), texts)}
ms["embeddings + nearest centre"] = ms["embeddings + LR"]
ms["SetFit"] = ms_per_message(setfit_models["en"].predict_proba, texts)
for r in rows:
    if r["method"] in ms:
        r["ms_per_message"] = round(ms[r["method"]], 2)
        r["run_per_1000"] = round(cpu_cost(ms[r["method"]]), 3)
        r["total_per_1000"] = round(r["mistakes_per_1000"] + r["run_per_1000"], 1)
comparison = table(rows)
comparison

A request whose confidence is below a threshold goes to the user: the app asks "Did you mean …?". Each question costs 0.10 dollars, but it avoids a 0.50- or 3-dollar mistake. The next cell finds the cheapest threshold on the validation requests for each candidate.

> **Check.** The cheapest candidate is TF-IDF + LR with all labels per intent: its cheapest threshold is 0.69, the app asks the user for 15.0% of the requests, and the total is about 38.3 dollars per 1,000. The zero-shot LLM, with no labels, is close behind.

In [ ]:
# Ask the user when the model is not sure: below a threshold, the request goes to "review".
def with_review(pred, conf, threshold):
    return pred.where(conf >= threshold, "review")

def best_threshold(name, k, part=None):
    """The threshold with the lowest total cost on the validation requests (steps of 0.01)."""
    found = []
    for th in np.round(np.arange(0.0, 1.0, 0.01), 2):
        r = score(name, validation, with_review(pred_val[name, k], conf_val[name, k], th), k)
        found.append({"threshold": th, "accuracy_of_decided": round(float(
            (pred_val[name, k] == validation.intent)[conf_val[name, k] >= th].mean()), 3) if (conf_val[name, k] >= th).any() else None,
            "review_share": r["review_share"], "mistakes_per_1000": r["mistakes_per_1000"]})
    found = pd.DataFrame(found)
    return found.loc[found.mistakes_per_1000.idxmin()], found

CANDIDATES = [("LLM zero-shot", 0), ("embeddings + LR", 10), ("SetFit", 10), ("LLM few-shot", 10), ("embeddings + LR", 50),
              ("TF-IDF + LR", 50), ("embeddings + LR", "all"), ("TF-IDF + LR", "all")]
choices, curves = [], {}
for name, k in CANDIDATES:
    best, curves[name, k] = best_threshold(name, k)
    run = next(r["run_per_1000"] for r in rows if r["method"].startswith(name) and r["labels_per_intent"] == k)
    choices.append({"method": name, "labels_per_intent": k, "threshold": best.threshold,
                    "review_share": best.review_share, "mistakes_per_1000": best.mistakes_per_1000,
                    "total_per_1000": round(best.mistakes_per_1000 + run, 1)})
choices = pd.DataFrame(choices)
CHOSEN, CHOSEN_K = "TF-IDF + LR", "all"   # the router we recommend (the page says why)
TH = float(choices.threshold[(choices.method == CHOSEN) & (choices.labels_per_intent == CHOSEN_K)].iloc[0])
choices

> **Your turn.** The product team accepts that the app asks the user at most 1 time in 10. Find the threshold with the fewest mistakes under that limit, from `found`.

In [ ]:
# The product team accepts that the app asks the user at most 1 time in 10.
found = curves[CHOSEN, CHOSEN_K]
capped = found[found.review_share <= 0.10].sort_values("mistakes_per_1000").iloc[0]
threshold_10 = float(capped.threshold)
print(f"threshold {threshold_10:.2f}: asks the user {capped.review_share:.1%}, mistakes {capped.mistakes_per_1000} per 1,000")

In [ ]:
expect('threshold_10', threshold_10, 0.59, tolerance=0.001)

## 14. When to stop labelling

Labels cost a person's time: 120 requests an hour at 28 dollars an hour (an **assumption**). The app gets 20,000 requests a month in each language (an **assumption**). For the cheapest model that runs on our own computer at each number of labels, the next cell computes the cost of the extra labels and how many months of savings pay for them. It also prints the hosted LLM's cost for comparison.

> **Check.** Going from 50 labels per intent to all pays back in about 1.4 months.

In [ ]:
# When to stop labelling? Labels cost a person's time; each step up the curve saves money every month.
LABEL_COST = 28 / 120          # dollars per label: 120 requests an hour at 28 dollars an hour (assumption)
MONTHLY = 20_000               # requests a month in each language (assumption)
size = {10: len(ids10), 50: len(draw(50)), "all": len(ALL)}
# the cheapest model that runs on our own computer (not the hosted LLM), with its threshold, at each number of labels
local = choices[~choices.method.str.startswith("LLM")]
steps = local.loc[local.groupby(local.labels_per_intent.astype(str), sort=False).total_per_1000.idxmin()].copy()
steps["labels_each_language"] = steps.labels_per_intent.map(size)
steps["label_cost"] = (steps.labels_each_language * 2 * LABEL_COST).round(0)   # both languages
steps["cost_per_month"] = (steps.total_per_1000 * MONTHLY * 2 / 1000).round(0)
steps["extra_label_cost"] = steps.label_cost.diff()
steps["saving_per_month"] = -steps.cost_per_month.diff()
steps["months_to_pay_back"] = (steps.extra_label_cost / steps.saving_per_month).round(1)
llm = choices[choices.method.str.startswith("LLM")].sort_values("total_per_1000").iloc[0]
print(f"For comparison, {llm.method} with {llm.labels_per_intent} labels per intent: {llm.total_per_1000} dollars "
      f"per 1,000, {llm.total_per_1000 * MONTHLY * 2 / 1000:,.0f} dollars a month, every request sent to a hosted service.")
cols = ["method", "labels_per_intent", "labels_each_language", "label_cost", "total_per_1000", "cost_per_month",
        "extra_label_cost", "saving_per_month", "months_to_pay_back"]
steps[cols]

## 15. The test, once

Now the test requests, for every candidate, with and without its threshold. Nothing changes after this cell.

> **Check.** The chosen router (TF-IDF + LR, all labels per intent, ask below 0.69) handles 0.814 of the test requests correctly without asking, asks the user for 14.8%, and costs about 50.2 dollars per 1,000. Test costs more than validation: thresholds chosen on 1,226 requests carry noise.

In [ ]:
# The test requests, used once, for the router we chose on the validation requests and the main alternatives.
def predict_test(name, k, part=None):
    part = test if part is None else part
    if name.startswith("LLM"):
        _, p, c = recorded(name.split(" ", 1)[1], part)
        return p, c
    if name == "SetFit":
        return setfit_predict(part)
    ids = {10: ids10, 50: draw(50), "all": ALL}[k]
    return per_language(METHODS[name], ids, part)

def predict_new(part):
    """The chosen router on new requests (not for the recorded LLM: it has no answers for them)."""
    return predict_test(CHOSEN, CHOSEN_K, part.reset_index(drop=True))

final = [score("everything to the user", test, ["review"] * len(test), 0)]
for _, c in choices.iterrows():
    run = next(r["run_per_1000"] for r in rows if r["method"].startswith(c.method) and r["labels_per_intent"] == c.labels_per_intent)
    p, conf = predict_test(c.method, c.labels_per_intent)
    final.append(score(f"{c.method}, {c.labels_per_intent} per intent", test, p, c.labels_per_intent, run))
    final.append(score(f"{c.method}, {c.labels_per_intent} per intent, ask below {c.threshold:.2f}", test,
                       with_review(p, conf, c.threshold), c.labels_per_intent, run))
final = table(final)
final[["method", "accuracy", "accuracy_en", "accuracy_fr", "rare_recall", "review_share", "total_per_1000"]]

## 16. New requests

We wrote six new requests, three in English and three in French. The router gives each one an intent and a confidence, and asks the user when it is not sure.

In [ ]:
# New requests that no model has seen (we wrote them for this page), through the chosen router.
new = pd.DataFrame({"language": ["en", "en", "en", "fr", "fr", "fr"], "text": [
    "wake me at 6 tomorrow, I have a flight",
    "did Sam answer my message about the budget",
    "take milk off the shopping list",
    "ajoute un rendez-vous chez le dentiste jeudi à 15 h",
    "est-ce que j'ai une alarme pour demain matin",
    "envoie un courriel à Anna pour dire que je serai en retard"]})
new["intent"], conf = predict_new(new)
new["confidence"] = conf.round(2)
new["decision"] = np.where(new.confidence >= TH, "do: " + new.intent, "ask the user")
new

Read each row. Do you agree with the intent? Would you want the app to ask?

## Next

The [lesson page](https://learning.nextia-ai.com/courses/text-routing/m10/sort-requests-with-few-labels/) explains each step, the costs, the recommendation and the limits, and how to make this case study a portfolio piece. Its knowledge checks count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Tag research papers with many labels](https://learning.nextia-ai.com/courses/text-routing/m10/tag-research-papers-with-many-labels/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/text-routing/m10/sort-requests-with-few-labels/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). MASSIVE is by Amazon, CC BY 4.0, built from SLURP (CC BY 4.0). Crowd workers wrote its requests for a voice assistant.